<a href="https://colab.research.google.com/github/Mojtaba-Choopani/LLM/blob/exercises/E%203-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1-1-Using a 10-sentence dataset, perform prompt engineering and In-Context Learning (ICL) without fine-tuning. Use a selected language model such as **Aya, Gemma, LLaMA, or another model** to translate English sentences into Persian. Evaluate the generated translations using the **BLEU score**.

In [2]:
data = [
    {
        "English": "I woke up early this morning.",
        "Persian": "من امروز صبح زود بیدار شدم."
    },
    {
        "English": "She is reading a very interesting book.",
        "Persian": "او دارد یک کتاب بسیار جالب می‌خواند."
    },
    {
        "English": "They went to the park to play football.",
        "Persian": "آن‌ها برای بازی فوتبال به پارک رفتند."
    },
    {
        "English": "We had dinner at a nice restaurant last night.",
        "Persian": "دیشب در یک رستوران خوب شام خوردیم."
    },
    {
        "English": "He doesn't like watching horror movies.",
        "Persian": "او تماشای فیلم‌های ترسناک را دوست ندارد."
    },
    {
        "English": "Can you help me with this math problem?",
        "Persian": "می‌تونی تو حل این مسئله ریاضی بهم کمک کنی؟"
    },
    {
        "English": "The weather is getting colder every day.",
        "Persian": "هوا هر روز سردتر می‌شود."
    },
    {
        "English": "I have never been to Paris.",
        "Persian": "من هرگز به پاریس نرفته‌ام."
    },
    {
        "English": "She always forgets where she puts her keys.",
        "Persian": "او همیشه فراموش می‌کند کلیدهایش را کجا گذاشته."
    },
    {
        "English": "We are planning a trip to the mountains.",
        "Persian": "ما داریم یک سفر به کوهستان برنامه‌ریزی می‌کنیم."
    }
]

In [1]:
# سلول ۱: نصب پکیج‌ها
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00


In [ ]:
# سلول ۲: بارگذاری مدل با کوانتایز 4-بیتی
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

In [4]:
def make_prompt(sentence):
    prompt = f"""
Translate the English sentence into natural Persian.
Only write the Persian translation.

Examples:

English: I am reading a book.
Persian: من دارم یک کتاب می‌خوانم.

English: The weather is cold today.
Persian: امروز هوا سرد است.

English: She went to school.
Persian: او به مدرسه رفت.

Now translate:

English: {sentence}
Persian:
"""
    return prompt

In [5]:
sentence = "I woke up early this morning."

prompt = make_prompt(sentence)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False
)

translation = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
).strip()

print(translation)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Translate the English sentence into natural Persian.
Only write the Persian translation.

Examples:

English: I am reading a book.
Persian: من دارم یک کتاب می‌خوانم.

English: The weather is cold today.
Persian: امروز هوا سرد است.

English: She went to school.
Persian: او به مدرسه رفت.

Now translate:

English: I woke up early this morning.
Persian:
من این صبح زود شستم.


In [ ]:
# سلول ۳: ساخت پرامپت با ICL (چند نمونه‌ی ترجمه‌شده به عنوان مثال)
few_shot_examples = """
English: The weather is beautiful today.
Persian: امروز هوا خیلی خوبه.

English: I would like a cup of coffee.
Persian: یه فنجان قهوه می‌خوام.

English: She works as a doctor in the city hospital.
Persian: او در بیمارستان شهر به عنوان پزشک کار می‌کند.
"""

sentences = [
    "The sun rises in the east.",
    "He loves playing football.",
    "This book is very interesting.",
    "We are going to the market tomorrow.",
    "The children are playing in the garden.",
    "I forgot my keys at home.",
    "The train arrives at nine o'clock.",
    "She sings beautifully.",
    "It is raining heavily outside.",
    "They traveled to Paris last summer.",
]

def build_prompt(sentence):
    return f"""شما یک مترجم حرفه‌ای انگلیسی به فارسی هستید. با توجه به نمونه‌های زیر، جمله‌ی جدید را ترجمه کنید.

{few_shot_examples}
English: {sentence}
Persian:"""

In [ ]:
# سلول ۴: اجرای ترجمه برای هر ۱۰ جمله
def translate(sentence):
    prompt = build_prompt(sentence)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    result = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return result.strip()

for i, s in enumerate(sentences, 1):
    translation = translate(s)
    print(f"{i}. {s}\n   → {translation}\n")